# 05 -- Final Load Preparation for Tableau

**Project:** Uber Data Visualization and Analysis  
**Input:** `data/processed/uber_trips_cleaned.csv`  
**Output:** `data/processed/uber_trips_tableau_ready.csv`  

Prepares the final Tableau-ready dataset: resolves Python-specific types, adds human-readable labels, computes final KPI columns, and exports the clean file.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == 'notebooks'
    else Path.cwd().resolve()
)

DATA_PATH    = PROJECT_ROOT / 'data' / 'processed' / 'uber_trips_cleaned.csv'
TABLEAU_PATH = PROJECT_ROOT / 'data' / 'processed' / 'uber_trips_tableau_ready.csv'

df = pd.read_csv(DATA_PATH, parse_dates=['pickup_time', 'drop_time'])

print(f'Loaded {len(df):,} rows x {df.shape[1]} columns')

Loaded 49,997 rows x 23 columns


## 1. Resolve Python-Specific Types

Tableau does not handle Python `bool`, `category`, or `date` objects well. All such columns are converted to plain strings or standard types.

In [2]:
# Boolean -> readable string
df['is_weekend'] = df['is_weekend'].map({True: 'Weekend', False: 'Weekday'})

# Category -> string
for col in df.select_dtypes('category').columns:
    df[col] = df[col].astype(str)

# pickup_date -> standard date string (YYYY-MM-DD)
df['pickup_date'] = pd.to_datetime(df['pickup_date']).dt.strftime('%Y-%m-%d')

print('Type conversions applied:')
print('  is_weekend     -> Weekday / Weekend (string)')
print('  category cols  -> string')
print('  pickup_date    -> YYYY-MM-DD string')

Type conversions applied:
  is_weekend     -> Weekday / Weekend (string)
  category cols  -> string
  pickup_date    -> YYYY-MM-DD string


## 2. Add Fare Tier Column

Precomputed tier labels are added so Tableau can filter by tier directly.

In [3]:
fare_q33 = df['fare_amount'].quantile(0.33)
fare_q66 = df['fare_amount'].quantile(0.66)

df['fare_tier'] = pd.cut(
    df['fare_amount'],
    bins=[-np.inf, fare_q33, fare_q66, np.inf],
    labels=['Budget', 'Standard', 'Premium'],
).astype(str)

print(f'Fare tier thresholds: Budget <= {fare_q33:.2f}, Standard <= {fare_q66:.2f}, Premium > {fare_q66:.2f}')
print(df['fare_tier'].value_counts())

Fare tier thresholds: Budget <= 12.87, Standard <= 18.09, Premium > 18.09
fare_tier
Premium     16989
Budget      16528
Standard    16480
Name: count, dtype: int64


## 3. Add Aggregated KPI Columns

City-level and overall KPIs are pre-aggregated and merged back so Tableau can reference them without LOD calculations.

In [4]:
# City-level KPIs
city_kpis = df.groupby('city').agg(
    city_total_trips=('trip_id', 'count'),
    city_avg_fare=('fare_amount', 'mean'),
    city_avg_distance=('distance_km', 'mean'),
    city_avg_duration=('trip_duration_mins', 'mean'),
    city_completion_rate=('status', lambda x: round((x == 'Completed').mean() * 100, 2)),
    city_cancellation_rate=('status', lambda x: round((x == 'Cancelled').mean() * 100, 2)),
    city_avg_fare_per_km=('fare_per_km', 'mean'),
).round(4).reset_index()

df = df.merge(city_kpis, on='city', how='left')

print('City-level KPI columns merged:')
print([c for c in df.columns if c.startswith('city_')])

City-level KPI columns merged:
['city_total_trips', 'city_avg_fare', 'city_avg_distance', 'city_avg_duration', 'city_completion_rate', 'city_cancellation_rate', 'city_avg_fare_per_km']


## 4. Final Dataset Validation

In [5]:
print('=' * 55)
print('  TABLEAU-READY DATASET -- FINAL SUMMARY')
print('=' * 55)
print(f'  Rows         : {len(df):,}')
print(f'  Columns      : {df.shape[1]}')
print(f'  Null values  : {df.isnull().sum().sum()}')
print(f'  Duplicates   : {df.duplicated().sum()}')
print(f'  Date range   : {df["pickup_date"].min()} to {df["pickup_date"].max()}')
print('=' * 55)

print('\nColumn list:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:>2}. {col:<30} {str(df[col].dtype)}')

  TABLEAU-READY DATASET -- FINAL SUMMARY
  Rows         : 49,997
  Columns      : 31
  Null values  : 0
  Duplicates   : 0
  Date range   : 2023-01-01 to 2023-02-04

Column list:
   1. trip_id                        int64
   2. driver_id                      int64
   3. rider_id                       int64
   4. city                           str
   5. pickup_lat                     float64
   6. pickup_lng                     float64
   7. drop_lat                       float64
   8. drop_lng                       float64
   9. distance_km                    float64
  10. fare_amount                    float64
  11. status                         str
  12. payment_method                 str
  13. pickup_time                    datetime64[us]
  14. drop_time                      datetime64[us]
  15. trip_duration_mins             float64
  16. pickup_date                    str
  17. pickup_hour                    int64
  18. pickup_day                     str
  19. pickup_month       

In [6]:
df.head(5)

,trip_id,driver_id,rider_id,city,pickup_lat,pickup_lng,drop_lat,drop_lng,distance_km,fare_amount,...,fare_per_km,is_weekend,fare_tier,city_total_trips,city_avg_fare,city_avg_distance,city_avg_duration,city_completion_rate,city_cancellation_rate,city_avg_fare_per_km
0,1,8270,10683,San Francisco,37.170931,-77.586479,37.173652,-77.619934,2.97,10.71,...,3.6061,Weekend,Budget,8401,16.0232,7.0309,21.0928,85.22,9.65,2.5971
1,2,1860,44743,Boston,38.898127,-108.582977,38.937464,-108.558727,8.43,22.41,...,2.6584,Weekend,Premium,8454,16.0521,7.0240,21.0721,84.95,10.33,2.6316
2,3,6390,75839,San Francisco,38.814571,-89.942603,38.821702,-89.896435,5.46,12.91,...,2.3645,Weekend,Standard,8401,16.0232,7.0309,21.0928,85.22,9.65,2.5971
3,4,6191,22189,New York,37.295906,-75.328844,37.301375,-75.317488,6.61,15.70,...,2.3752,Weekend,Standard,8247,16.0485,7.0141,21.0424,85.32,9.81,2.6019
4,5,6734,61104,Seattle,38.972395,-121.482913,38.992088,-121.467904,10.50,19.15,...,1.8238,Weekend,Premium,8236,16.0621,7.0685,21.2055,85.45,9.79,2.6697


## 5. Export Tableau-Ready Dataset

In [7]:
TABLEAU_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(TABLEAU_PATH, index=False)

print(f'Tableau-ready dataset exported to:')
print(f'  {TABLEAU_PATH}')
print(f'  Rows: {len(df):,}  |  Columns: {df.shape[1]}')

Tableau-ready dataset exported to:
  /Users/ravleensingh/Documents/Capstone/DVA_Capstone/Uber_Analysis/data/processed/uber_trips_tableau_ready.csv
  Rows: 49,997  |  Columns: 31


---

## Pipeline Complete

| File | Location | Purpose |
|---|---|---|
| `uber_trips_cleaned.csv` | `data/processed/` | Analysis-ready dataset |
| `uber_trips_tableau_ready.csv` | `data/processed/` | Direct Tableau ingestion |

**Next Step:** Build and publish the Tableau dashboard using `uber_trips_tableau_ready.csv`.